In [1]:
import sys
sys.path.append("..")

import lightgbm
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.pipeline import Pipeline

from src.features import build_features, get_numeric_features, CATEGORICAL_FEATURES, BINARY_FEATURES
from src.models import build_lightgbm_preprocessor, build_lightgbm_model, CalibratedModel, logit

train = pd.read_parquet("../data/processed/train.parquet")
valid = pd.read_parquet("../data/processed/valid.parquet")
test = pd.read_parquet("../data/processed/test.parquet")

X_train, y_train = train.drop(columns=["target"]), train["target"]
X_valid, y_valid = valid.drop(columns=["target"]), valid["target"]
X_test, y_test = test.drop(columns=["target"]), test["target"]

numeric_features = get_numeric_features(build_features(train.head(1000)))

CHOSEN_PARAMS = {
    "num_leaves": 15,
    "min_child_samples": 500,
    "cat_smooth": 100,
    "min_data_per_group": 1000,
}

In [2]:
def train_lightgbm(categorical):
    prep = build_lightgbm_preprocessor(numeric_features, categorical, BINARY_FEATURES)
    X_tr = prep.fit_transform(X_train)
    X_va = prep.transform(X_valid)

    model = build_lightgbm_model(**CHOSEN_PARAMS)
    model.fit(
        X_tr, y_train,
        eval_X=X_va,
        eval_y=y_valid,
        eval_metric="auc",
        categorical_feature=categorical,
        callbacks=[lightgbm.early_stopping(stopping_rounds=100, verbose=False)],
    )
    return Pipeline([("prep", prep), ("model", model)])


no_state = [c for c in CATEGORICAL_FEATURES if c != "addr_state"]

with_state = train_lightgbm(CATEGORICAL_FEATURES)
without_state = train_lightgbm(no_state)
print("Both models trained.")

Both models trained.


In [3]:
def calibrate(pipeline):
    raw_valid = pipeline.predict_proba(X_valid)[:, 1]
    calibrator = LogisticRegression()
    calibrator.fit(logit(raw_valid).reshape(-1, 1), y_valid)
    return CalibratedModel(pipeline, calibrator)


models = {
    "With state": calibrate(with_state),
    "Without state": calibrate(without_state),
}

p_valid = {name: m.predict_proba(X_valid)[:, 1] for name, m in models.items()}
p_test = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}

for name, m in models.items():
    trees = m.pipeline.named_steps["model"].best_iteration_
    auc = roc_auc_score(y_valid, p_valid[name])
    print(f"{name}: {trees} trees, validation AUC {auc:.4f}")

With state: 813 trees, validation AUC 0.6951
Without state: 680 trees, validation AUC 0.6932


In [4]:
def ks_statistic(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return (tpr - fpr).max()


test_table = pd.DataFrame({
    name: {
        "auc": roc_auc_score(y_test, p),
        "ks": ks_statistic(y_test, p),
        "brier": brier_score_loss(y_test, p),
        "avg_predicted": p.mean(),
    }
    for name, p in p_test.items()
}).T
test_table["gini"] = 2 * test_table["auc"] - 1

print("Actual test default rate:", round(y_test.mean(), 4))
test_table.round(4)

Actual test default rate: 0.1471


,auc,ks,brier,avg_predicted,gini
With state,0.7026,0.2941,0.1170,0.1462,0.4051
Without state,0.7005,0.2927,0.1172,0.1459,0.4009


In [5]:
rng = np.random.default_rng(42)
y_arr = y_test.values
p_with, p_without = p_test["With state"], p_test["Without state"]

differences = []
for _ in range(200):
    sample = rng.integers(0, len(y_arr), size=len(y_arr))
    differences.append(
        roc_auc_score(y_arr[sample], p_with[sample])
        - roc_auc_score(y_arr[sample], p_without[sample])
    )

differences = np.array(differences)
low, high = np.percentile(differences, [2.5, 97.5])
print(f"AUC gain from using state: {differences.mean():.4f}")
print(f"95% range: {low:.4f} to {high:.4f}")
print(f"With state ahead in {(differences > 0).mean() * 100:.0f}% of samples")

AUC gain from using state: 0.0021
95% range: 0.0012 to 0.0032
With state ahead in 100% of samples


In [6]:
def state_gaps(p, min_loans=1000):
    df = pd.DataFrame({
        "state": test["addr_state"].values,
        "actual": y_test.values,
        "predicted": p,
    })
    g = df.groupby("state").agg(
        loans=("actual", "size"),
        actual_pct=("actual", "mean"),
        predicted_pct=("predicted", "mean"),
    )
    g = g[g["loans"] >= min_loans].copy()
    g[["actual_pct", "predicted_pct"]] *= 100
    g["gap_pct"] = g["predicted_pct"] - g["actual_pct"]
    return g


gaps = {name: state_gaps(p) for name, p in p_test.items()}

for name, g in gaps.items():
    print(f"{name}: {len(g)} states, average absolute gap "
          f"{g['gap_pct'].abs().mean():.2f} points, largest {g['gap_pct'].abs().max():.2f}")

comparison = pd.DataFrame({
    "loans": gaps["With state"]["loans"],
    "actual_pct": gaps["With state"]["actual_pct"],
    "gap_with_state": gaps["With state"]["gap_pct"],
    "gap_without_state": gaps["Without state"]["gap_pct"],
})
order = comparison["gap_with_state"].abs().sort_values(ascending=False).index
comparison.loc[order].head(10).round(2)

With state: 33 states, average absolute gap 1.03 points, largest 3.54
Without state: 33 states, average absolute gap 1.71 points, largest 4.73


,loans,actual_pct,gap_with_state,gap_without_state
state,,,,
AR,1214,20.10,-3.54,-4.63
OR,1795,9.92,2.84,4.73
OK,1451,19.37,-2.55,-4.47
LA,1858,17.60,-2.09,-2.67
SC,1818,10.73,1.98,4.51
WI,2134,11.72,1.95,2.74
VA,4314,13.40,1.50,0.14
KY,1509,15.64,-1.41,-0.63
NJ,5826,16.13,-1.40,-2.26


In [7]:
profit_test = (test["total_pymnt"] - test["funded_amnt"]).values
profit_valid = (valid["total_pymnt"] - valid["funded_amnt"]).values
thresholds = np.round(np.arange(0.05, 0.605, 0.005), 3)


def best_cutoff(p, profit_values):
    totals = [profit_values[p < t].sum() for t in thresholds]
    return thresholds[int(np.argmax(totals))]


money = {}
for name in models:
    cutoff = best_cutoff(p_valid[name], profit_valid)
    approved = p_test[name] < cutoff
    money[name] = {
        "cutoff": cutoff,
        "approval_rate_pct": approved.mean() * 100,
        "profit_millions": profit_test[approved].sum() / 1e6,
        "gain_vs_approve_all": (profit_test[approved].sum() - profit_test.sum()) / 1e6,
    }

pd.DataFrame(money).T.round(3)

,cutoff,approval_rate_pct,profit_millions,gain_vs_approve_all
With state,0.36,97.183,134.750,2.552
Without state,0.36,97.361,134.697,2.499


In [8]:
summary = test_table.join(pd.DataFrame(money).T)
summary["avg_abs_state_gap"] = [gaps[name]["gap_pct"].abs().mean() for name in summary.index]

summary.round(4).to_csv("../results/state_comparison.csv")
summary.round(4)

,auc,ks,brier,avg_predicted,gini,cutoff,approval_rate_pct,profit_millions,gain_vs_approve_all,avg_abs_state_gap
With state,0.7026,0.2941,0.1170,0.1462,0.4051,0.36,97.1831,134.7505,2.5517,1.0283
Without state,0.7005,0.2927,0.1172,0.1459,0.4009,0.36,97.3613,134.6974,2.4986,1.7092
